# Compare Local SDK Workflows

Run one GHZ workload through every installed core SDK adapter. This replaces separate per-SDK notebooks with one comparison-oriented workflow.

## Variables and Parameters

- `benchmark`: the shared three-qubit GHZ specification.
- `backends`: installed core execution adapters selected from Cirq, Qiskit Aer, PennyLane, and Braket LocalSimulator.
- `shots`: 128 samples per SDK.
- `results`: standardized rows returned by the package runner.


In [ ]:
from quantum_backend_bench.core.circuit_export import export_benchmark_circuit
from quantum_backend_bench.core.discovery import backend_capabilities
from quantum_backend_bench.core.factory import build_benchmark_from_config
from quantum_backend_bench.core.runner import run_benchmark
from quantum_backend_bench.utils.formatting import format_results_table
from quantum_backend_bench.utils.notebook import (
    notebook_artifact_dir,
    save_result_artifacts,
    top_measurement_states,
    verification_frame,
)

ARTIFACT_DIR = notebook_artifact_dir()
CORE_BACKENDS = ("cirq", "qiskit_aer", "pennylane", "braket_local")
benchmark = build_benchmark_from_config({"benchmark": "ghz", "n_qubits": 3})
installed = {item.name for item in backend_capabilities() if item.installed}
backends = [name for name in CORE_BACKENDS if name in installed]
shots = 128

## Neutral Export

The same neutral benchmark is exported before any SDK-specific execution.


In [ ]:
print(export_benchmark_circuit(benchmark, "openqasm"))

## Execute and Verify

Each installed adapter receives the same benchmark and shot count. Artifacts retain stable per-SDK names for documentation builds.


In [ ]:
if not backends:
    raise RuntimeError("Install at least one core SDK extra before running this notebook.")

results = run_benchmark(benchmark, backends, shots=shots)
print(format_results_table(results))
display({result["backend"]: top_measurement_states(result) for result in results})
display(verification_frame(results))

artifact_stems = {
    "cirq": "sdk_cirq_workflow",
    "qiskit_aer": "sdk_qiskit_workflow",
    "pennylane": "sdk_pennylane_workflow",
    "braket_local": "sdk_braket_workflow",
}
for result in results:
    save_result_artifacts([result], artifact_stems[result["backend"]], ARTIFACT_DIR)